# Prática — Módulo 4 – Representação Vetorial Baseada em Coocorrência

## Prática
Nesta prática o aluno constrói passo a passo diferentes representações vetoriais a partir de um pequeno corpus textual. Inicialmente são exploradas estatísticas básicas do corpus, seguida da geração de matrizes BoW e TF-IDF. Em seguida são extraídos bigramas sequenciais para construir uma matriz de coocorrência palavra-contexto. A partir dessa matriz são calculadas probabilidades conjuntas, probabilidades marginais e, por fim, os valores de PMI e PPMI, permitindo observar como relações estatísticas entre palavras emergem diretamente dos dados.

## Dataset
O corpus utilizado consiste em um conjunto pequeno de frases sintéticas envolvendo agentes (como gato, cachorro e homem), verbos de ação (bebe, come) e alimentos ou líquidos (leite, água, carne, peixe). Embora simples, esse corpus é suficiente para ilustrar como padrões de coocorrência podem revelar associações entre palavras, como verbos e objetos, permitindo visualizar o funcionamento das matrizes de coocorrência e das medidas de associação utilizadas em PLN.




In [14]:
import subprocess
import sys
import importlib

packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "nltk": "nltk",
}

for module, package in packages.items():
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [7]:
# Importar bibliotecas
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [8]:
# Definir um pequeno corpus para analisar coocorrência entre palavras.

# Corpus minúsculo.
corpus = [
    "gato bebe agua",
    "cachorro bebe agua",
    "gato bebe leite",
    "cachorro bebe leite",
    "gato come peixe",
    "cachorro come peixe",
    "gato come carne",
    "cachorro come carne",
    "homem bebe agua",
    "homem come carne",
]

corpus

# Visualizar o corpus.
for i, frase in enumerate(corpus, 1):
    print(f"D{i}: {frase}")

D1: gato bebe agua
D2: cachorro bebe agua
D3: gato bebe leite
D4: cachorro bebe leite
D5: gato come peixe
D6: cachorro come peixe
D7: gato come carne
D8: cachorro come carne
D9: homem bebe agua
D10: homem come carne


## PARTE 1 — BoW

### Pergunta
1. Quantas palavras diferentes existem no corpus?
    
    Existem 9 palavras diferentes (únicas) no vocabulário do corpus.

2. Quais palavras aparecem em mais de um documento?
   
    Todas as 9 palavras do corpus aparecem em mais de um documento.

In [10]:
# Contar palavras no corpus.
from collections import Counter

tokens = " ".join(corpus).split()

freq = Counter(tokens)
freq.most_common()


[('bebe', 5),
 ('come', 5),
 ('gato', 4),
 ('cachorro', 4),
 ('agua', 3),
 ('carne', 3),
 ('leite', 2),
 ('peixe', 2),
 ('homem', 2)]

In [11]:
# Criar matriz Bag of Words.
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(corpus)

bow = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"D{i+1}" for i in range(len(corpus))]
)

bow

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
D1,1,1,0,0,0,1,0,0,0
D2,1,1,1,0,0,0,0,0,0
D3,0,1,0,0,0,1,0,1,0
D4,0,1,1,0,0,0,0,1,0
D5,0,0,0,0,1,1,0,0,1
D6,0,0,1,0,1,0,0,0,1
D7,0,0,0,1,1,1,0,0,0
D8,0,0,1,1,1,0,0,0,0
D9,1,1,0,0,0,0,1,0,0
D10,0,0,0,1,1,0,1,0,0


### Pergunta
3. Quantos vetores existem na matriz BoW?
    
    Existem 10 vetores na matriz BoW (um vetor para cada frase de D1 até D10).

4. Cada vetor representa o quê?
    
    Cada vetor representa um documento único (neste caso, uma frase específica do corpus), modelado sob a ótica da frequência de ocorrência das palavras do vocabulário nesse documento.

5. Qual é a dimensão de cada vetor?
    
    A dimensão de cada vetor é 9.

6. A dimensão do vetor depende de quê?
    
    A dimensão do vetor no modelo Bag of Words depende do tamanho total do vocabulário, ou seja, da quantidade de palavras únicas extraídas da união de todos os documentos presentes no corpus.

7. Documentos que compartilham palavras terão vetores parecidos?
    
    Sim. Como os vetores registram as ocorrências e contagens de palavras, documentos que possuem vocabulários em comum terão os mesmos índices ativados com valores próximos, o que resultará em uma maior semelhança geométrica (alta similaridade de cosseno, por exemplo) no espaço vetorial.

8. Pela matriz BoW, é possível perceber que “bebe” aparece tanto com “gato” quanto com “cachorro”? Ou essa relação não fica explícita na matriz?
    
    Olhando os vetores individualmente, podemos observar essas coocorrências no nível de documento. No entanto, a relação sintática de ordem ou associação direta é perdida pelo fato do modelo ser apenas um "saco de palavras". A matriz não diz que o gato está bebendo, apenas que as palavras dividem a mesma frase.

PARTE 2 — TF-IDF

### Pergunta
9. A matriz TF-IDF possui o mesmo número de vetores do BoW?
    
    Sim, possui os mesmos 10 vetores, já que a base de documentos é a mesma.

10. A dimensão dos vetores mudou em relação ao BoW?
    
    Não, a dimensão continua sendo 9, pois o vocabulário base utilizado para gerar as colunas se mantém.

11. Por que agora aparecem valores decimais em vez de contagens inteiras?
    
    Porque o TF-IDF calcula um peso para cada palavra com base na sua frequência no documento (TF) ajustada pela sua raridade em todo o corpus (IDF). Além disso, o TfidfVectorizer aplica nativamente uma normalização matemática (Normalização L2) nos vetores, dividindo-os para que tenham comprimento unitário, o que os transforma em valores float contínuos entre 0 e 1.

12. Todas as palavras possuem o mesmo peso dentro de um documento?
    
    Não. O peso varia, pois leva em conta a relevância. Palavras que aparecem em muitos documentos no corpus geral acabam sendo "penalizadas" e ganham um peso ligeiramente inferior em comparação a substantivos mais raros que ajudam melhor a discriminar o assunto daquela frase.

13. Qual é a palavra mais informativa (mais discriminativa) de cada documento?
    
    É a palavra que possui a maior pontuação (peso) na linha do TF-IDF do respectivo documento:

    D1: agua (0.641)

    D2: agua (0.641)

    D3: leite (0.691)

    D4: leite (0.691)

    D5: peixe (0.691)

    D6: peixe (0.691)

    D7: carne (0.641)

    D8: carne (0.641)

    D9: homem (0.666)

    D10: homem (0.666)


In [15]:
# gerar matriz TF-IDF.
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(corpus)

tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"D{i+1}" for i in range(len(corpus))]
)

tfidf

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
D1,0.641771,0.512414,0.000000,0.000000,0.000000,0.570581,0.000000,0.000000,0.000000
D2,0.641771,0.512414,0.570581,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
D3,0.000000,0.482845,0.000000,0.000000,0.000000,0.537655,0.000000,0.691222,0.000000
D4,0.000000,0.482845,0.537655,0.000000,0.000000,0.000000,0.000000,0.691222,0.000000
D5,0.000000,0.000000,0.000000,0.000000,0.482845,0.537655,0.000000,0.000000,0.691222
D6,0.000000,0.000000,0.537655,0.000000,0.482845,0.000000,0.000000,0.000000,0.691222
D7,0.000000,0.000000,0.000000,0.641771,0.512414,0.570581,0.000000,0.000000,0.000000
D8,0.000000,0.000000,0.570581,0.641771,0.512414,0.000000,0.000000,0.000000,0.000000
D9,0.582818,0.465343,0.000000,0.000000,0.000000,0.000000,0.666168,0.000000,0.000000
D10,0.000000,0.000000,0.000000,0.582818,0.465343,0.000000,0.666168,0.000000,0.000000


### Pergunta
14. O TF-IDF consegue capturar relação entre palavras como bebe → leite?
    
    Não. Embora o TF-IDF ajuste os pesos das palavras, ele continua seguindo a abordagem de independência posicional do Bag of Words. A ordem sequencial ou a relação semântica (verbo-objeto) entre bebe e leite não é capturada nesse modelo.

## PARTE 3 — Coocorrência (n-gramas)

In [16]:
# Gerar bigramas sequenciais (janela de contexto = "1 à direita").
from nltk.util import bigrams

bigrams_list = []

for sent in corpus:
    tokens = sent.split()
    bigrams_list.extend(list(bigrams(tokens)))

bigrams_list

[('gato', 'bebe'),
 ('bebe', 'agua'),
 ('cachorro', 'bebe'),
 ('bebe', 'agua'),
 ('gato', 'bebe'),
 ('bebe', 'leite'),
 ('cachorro', 'bebe'),
 ('bebe', 'leite'),
 ('gato', 'come'),
 ('come', 'peixe'),
 ('cachorro', 'come'),
 ('come', 'peixe'),
 ('gato', 'come'),
 ('come', 'carne'),
 ('cachorro', 'come'),
 ('come', 'carne'),
 ('homem', 'bebe'),
 ('bebe', 'agua'),
 ('homem', 'come'),
 ('come', 'carne')]

In [17]:
# Contar frequência dos bigramas.
from collections import Counter

bigram_freq = Counter(bigrams_list)
bigram_freq

Counter({('bebe', 'agua'): 3,
         ('come', 'carne'): 3,
         ('gato', 'bebe'): 2,
         ('cachorro', 'bebe'): 2,
         ('bebe', 'leite'): 2,
         ('gato', 'come'): 2,
         ('come', 'peixe'): 2,
         ('cachorro', 'come'): 2,
         ('homem', 'bebe'): 1,
         ('homem', 'come'): 1})

In [18]:
# Criar matriz de coocorrência a partir dos bigramas.
words = sorted(set([w for bg in bigram_freq for w in bg]))

matrix = pd.DataFrame(0, index=words, columns=words)

for (w,c),freq in bigram_freq.items():
    matrix.loc[w,c] = freq

matrix

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,0,0,0,0,0,0,0,0,0
bebe,3,0,0,0,0,0,0,2,0
cachorro,0,2,0,0,2,0,0,0,0
carne,0,0,0,0,0,0,0,0,0
come,0,0,0,3,0,0,0,0,2
gato,0,2,0,0,2,0,0,0,0
homem,0,1,0,0,1,0,0,0,0
leite,0,0,0,0,0,0,0,0,0
peixe,0,0,0,0,0,0,0,0,0


## PARTE 4 — PPMI

### Pergunta
15. O que significa a probabilidade conjunta P(w,c) para um par de palavras?

    Significa a probabilidade de encontrarmos o par exato (palavra_alvo, palavra_contexto) juntos em relação ao volume total de ocorrências extraídos do corpus de texto.


In [19]:
# Calcular total de coocorrências.
N = matrix.values.sum()
N

np.int64(20)

In [20]:
# Calcular probabilidade conjunta dos bigramas.
N = sum(bigram_freq.values())

p_wc = {bg: freq / N for bg, freq in bigram_freq.items()}
p_wc

{('gato', 'bebe'): 0.1,
 ('bebe', 'agua'): 0.15,
 ('cachorro', 'bebe'): 0.1,
 ('bebe', 'leite'): 0.1,
 ('gato', 'come'): 0.1,
 ('come', 'peixe'): 0.1,
 ('cachorro', 'come'): 0.1,
 ('come', 'carne'): 0.15,
 ('homem', 'bebe'): 0.05,
 ('homem', 'come'): 0.05}

### Pergunta
16. Qual é o valor da soma das probabilidades conjuntas? O resultado seria o mesmo independentemente da quantidade de palavras?

    A soma de todas as probabilidades na matriz é igual a 1 (ou 100%). Sim, o resultado sempre será 1 independentemente do tamanho do corpus ou da quantidade de palavras, pois a divisão garante uma distribuição de probabilidade perfeitamente normalizada sobre todo o conjunto de dados.

In [21]:
# Gerar matrix de probabilidade conjunta.
N = matrix.values.sum()

p_wc = matrix / N
p_wc

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0
bebe,0.15,0.00,0.0,0.00,0.00,0.0,0.0,0.1,0.0
cachorro,0.00,0.10,0.0,0.00,0.10,0.0,0.0,0.0,0.0
carne,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0
come,0.00,0.00,0.0,0.15,0.00,0.0,0.0,0.0,0.1
gato,0.00,0.10,0.0,0.00,0.10,0.0,0.0,0.0,0.0
homem,0.00,0.05,0.0,0.00,0.05,0.0,0.0,0.0,0.0
leite,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0
peixe,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0


### Pergunta
17. Se P(w,c) é alto, por si só, isso implica forte associação entre w e c?

    Não obrigatoriamente. Valores absolutos altos podem ocorrer apenas porque as palavras individuais w e c aparecem de forma absurdamente frequente no corpus todo (como artigos "o", "a", preposições "de", "com"), mesmo que não haja uma ligação fundamental significativa entre elas no idioma.

18. Como saber se duas palavras aparecem juntas mais do que seria esperado ao acaso?

    Calculando o PMI (Pointwise Mutual Information). Ele calcula a razão entre a probabilidade de ambas aparecerem juntas P(w,c) e a probabilidade teórica de aparecerem juntas se fossem completamente independentes, dado por P(w) * P(c). Se esse valor (após logaritmo) for maior que 0, significa que elas coocorrem mais do que ditaria o mero acaso estatístico.

In [22]:
# Calcular frequência marginal das palavras.
p_w = matrix.sum(axis=1) / N
p_w

agua        0.00
bebe        0.25
cachorro    0.20
carne       0.00
come        0.25
gato        0.20
homem       0.10
leite       0.00
peixe       0.00
dtype: float64

In [23]:
# Calcular frequência marginal dos contextos.
p_c = matrix.sum(axis=0) / N
p_c

agua        0.15
bebe        0.25
cachorro    0.00
carne       0.15
come        0.25
gato        0.00
homem       0.00
leite       0.10
peixe       0.10
dtype: float64

In [24]:
# Calcular PMI (Pointwise Mutual Information).
import numpy as np

pmi = np.log(p_wc / (p_w.values[:, None] * p_c.values[None, :]))
pmi = pmi.replace([np.inf, -np.inf], 0)
pmi

c:\_Pessoas\Bruno.Araujo\CDN_PLN\venvPln\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bebe,1.386294,0.000000,NaN,0.000000,0.000000,NaN,NaN,1.386294,0.000000
cachorro,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
carne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
come,0.000000,0.000000,NaN,1.386294,0.000000,NaN,NaN,0.000000,1.386294
gato,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
homem,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
leite,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
peixe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Pergunta
19. Porque apareceu NaN?

    Os "NaN" ocorreram no passo do cálculo PMI porque muitos pares de palavras nunca aparecem juntos no corpus. Isso faz com que a probabilidade conjunta P(w,c) seja zero. Como a fórmula matemática do PMI aplica um logaritmo na probabilidade, log(0) tende a infinito negativo o que resulta em indefinição.

20. O que acontece no passo chamado Positive PMI?

    A métrica Positive PMI (PPMI) substitui todos os logaritmos que resultaram em valores negativos ou erros (NaN, -inf) por 0. A justificativa teórica é que não interessa muito mensurar "o quão pouco" palavras se associam; o foco computacional é apenas nas relações que coocorrem com força maior que a média (associações positivas).

In [25]:
# Converter para PPMI.
ppmi = pmi.clip(lower=0).fillna(0)
ppmi

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
bebe,1.386294,0.000000,0.0,0.000000,0.000000,0.0,0.0,1.386294,0.000000
cachorro,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
carne,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
come,0.000000,0.000000,0.0,1.386294,0.000000,0.0,0.0,0.000000,1.386294
gato,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
homem,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
leite,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
peixe,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000


### Pergunta
21. A matriz PPMI indica claramente quais alimentos estão associados a bebe e quais estão associados a come. No entanto, é possível determinar a partir dela qual agente (gato, cachorro, homem) bebe ou come cada alimento específico?

    Não é possível. Como os bigramas foram criados com uma "janela = 1" estrita (apenas olhando o próximo vizinho), a matriz registrou a relação (agente → verbo) e separadamente a relação (verbo → alimento). Por exemplo, sabemos que cachorro se associa a come e que come se associa a carne. Mas a estrutura da matriz bi-dimensional PPMI que geramos não mantém o registro transitivo que formaria o trigrama "cachorro come carne", perdendo quem é o sujeito exato de cada alimento.

22. Esse tipo de identificação de associação entre verbos e objetos (por exemplo, bebe–água ou come–peixe) era possível utilizando apenas BoW ou TF-IDF? Explique por quê.

    Não. Os métodos BoW e TF-IDF trabalham apenas com a "bolsa" agregada de palavras de um documento, desconstruindo e descartando a estrutura gramatical original. Por conta disso, eles tratam todas as ocorrências como vizinhas simultâneas da frase inteira. Já a matriz PPMI montada a partir de coocorrência (n-gramas sequenciais) olha as posições relativas imediatas entre os tokens.